# DivergenceMagnetique — `divergence_magnetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **differential** post-treatment fields of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `divergence_magnetique`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$
\nabla \cdot \mathbf{b} = \operatorname{tr}(\nabla b) = \sum_i \frac{\partial b_i}{\partial x_i}
$$
Exactly linear field $b = (x, y)$: $\nabla b = \mathbb{1}$, so $\nabla\cdot b = 2$ in
every cell (Green-Gauss is exact for linear fields).


## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | magnetic field divergence | $\nabla\cdot b = \operatorname{tr}(\nabla b)$ |

All these fields are built from the module-reconstructed element gradient
$\nabla b$ (`gradient_b_elements`, Green-Gauss — the kernel has no
element-field gradient factory on PolyMAC). Vector model required: the
scalar model is rejected with a clean error by `creer_champ`.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).



In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "divergence_magnetique.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition divergence_magnetique.data 2 -> PAR_divergence_magnetique.data + DOM_000*.Zones
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)



## Checks


In [ ]:
t, div = read_son(case.dataFileName_, "SONDE_DIV")

# b = (x, y) is exactly linear => div b = 1 + 1 = 2 at every cell centre.
expected = 2.0
print(f"divergence_magnetique (probe) : {div[-1]:.6g}  (expected {expected:.6g})")
err = abs(div[-1] - expected) / expected
print(f"relative error                : {err:.3e}")
assert err < 1e-12, f"divergence_magnetique wrong: {div[-1]}"
print("OK: divergence_magnetique = trace(grad b).")

# --- Boundary-cell check (B2: one-sided reconstruction) ---
t, div_bd = read_son(case.dataFileName_, "SONDE_DIV_BD")
print(f"divergence_magnetique (boundary): {div_bd[-1]:.6g}  (expected {expected:.6g})")
err_bd = abs(div_bd[-1] - expected) / expected
print(f"relative error (boundary)      : {err_bd:.3e}")
assert err_bd < 1e-12, f"boundary divergence_magnetique wrong: {div_bd[-1]}"
print("OK: boundary cell also recovers exact div b with B2 one-sided correction.")


## Computer Performance


In [ ]:
run.tablePerf()

